# 09 — Validacija konačnog integrisanog profila

Završni eksperiment proverava preporučeni profil kao celinu nad svih 40 snimaka čitanog srpskog govora:

**Faster-Whisper Small + INT8 + beam_size=5 + VAD ON + chunk=10 s**

Za svaki snimak računaju se WER, vreme obrade i RTF, a na kraju se računaju i prosečan WER, medijana WER-a i corpus WER.

**VAŽNO:** u ćeliju `test_set` treba prekopirati isti skup od 40 snimaka i iste referentne transkripte iz postojećeg benchmark notebook-a. Reference se ne menjaju.


In [1]:
from pathlib import Path
import time
import re
import unicodedata

import numpy as np
import pandas as pd
from pydub import AudioSegment
from faster_whisper import WhisperModel
from jiwer import wer


In [2]:
# ============================================================
# PUTANJE I KONAČNA KONFIGURACIJA
# ============================================================

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

AUDIO_DIR = PROJECT_ROOT / "audio"
REZULTATI_DIR = PROJECT_ROOT / "rezultati" / "09_validacija_konacnog_profila"
REZULTATI_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SIZE = "small"
COMPUTE_TYPE = "int8"
DEVICE = "cpu"

LANGUAGE = "sr"
BEAM_SIZE = 5
VAD_FILTER = True
CHUNK_SECONDS = 10

print("PROJECT_ROOT:", PROJECT_ROOT)
print("AUDIO_DIR:", AUDIO_DIR)
print("REZULTATI_DIR:", REZULTATI_DIR)

print("\nKONAČNI PROFIL:")
print(f"Model:          {MODEL_SIZE}")
print(f"Kvantizacija:   {COMPUTE_TYPE}")
print(f"Uređaj:         {DEVICE}")
print(f"Jezik:          {LANGUAGE}")
print(f"Beam size:      {BEAM_SIZE}")
print(f"VAD:            {VAD_FILTER}")
print(f"Chunk:          {CHUNK_SECONDS} s")


PROJECT_ROOT: D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR
AUDIO_DIR: D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR\audio
REZULTATI_DIR: D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR\rezultati\09_validacija_konacnog_profila

KONAČNI PROFIL:
Model:          small
Kvantizacija:   int8
Uređaj:         cpu
Jezik:          sr
Beam size:      5
VAD:            True
Chunk:          10 s


## Test skup

U sledeću ćeliju prekopirati **isti `test_set` od 40 snimaka sa istim referencama** koji je korišćen u prethodnim benchmark eksperimentima.

Ako postojeći `test_set` već sadrži ključeve `audio` i `ref`, može se prekopirati bez drugih izmena.


In [3]:
# ============================================================
# TEST SET — PREKOPIRATI ISTIH 40 STAVKI IZ POSTOJEĆEG BENCHMARKA
# ============================================================

test_set = [
    # PRIMER STRUKTURE:
    # {
    #     "broj": 1,
    #     "audio": AUDIO_DIR / "1govornik1.wav",
    #     "ref": "Ovde ide postojeći referentni tekst."
    # },

    # OVDE PREKOPIRATI SVIH 40 POSTOJEĆIH STAVKI.
]

assert len(test_set) == 40, (
    f"Očekivano 40 snimaka, pronađeno {len(test_set)}. "
    "Prekopiraj kompletan postojeći test_set."
)

for item in test_set:
    putanja = Path(item["audio"])
    if not putanja.exists():
        raise FileNotFoundError(f"Nedostaje audio: {putanja}")

print(f"✓ Učitano {len(test_set)} snimaka.")


AssertionError: Očekivano 40 snimaka, pronađeno 0. Prekopiraj kompletan postojeći test_set.

In [ ]:
# ============================================================
# NORMALIZACIJA I POMOĆNE FUNKCIJE
# ============================================================

def normalizuj_tekst(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKC", text)

    # Interpunkcija -> razmak
    text = re.sub(r"[^\w\sčćžšđ]", " ", text, flags=re.UNICODE)

    # Višestruki razmaci -> jedan
    text = re.sub(r"\s+", " ", text).strip()

    return text


def govornik_iz_imena(putanja):
    ime = Path(putanja).stem.lower()

    if "govornik1" in ime:
        return "G1"
    elif "govornik2" in ime:
        return "G2"
    elif "govornik3" in ime:
        return "G3"

    return "Nepoznato"


def trajanje_audio(putanja):
    audio = AudioSegment.from_file(str(putanja))
    return len(audio) / 1000.0


In [ ]:
# ============================================================
# UČITAVANJE FINALNOG MODELA
# ============================================================

print("Učitavanje finalnog modela...")

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)

print("✓ Model učitan.")


In [ ]:
# ============================================================
# TRANSKRIPCIJA U SEGMENTIMA OD 10 SEKUNDI
# ============================================================

def transkribuj_u_chunkovima(
    model,
    audio_path,
    chunk_seconds=10,
    language="sr",
    beam_size=5,
    vad_filter=True
):
    """
    Deli audio zapis na segmente maksimalnog trajanja chunk_seconds,
    transkribuje svaki segment istom konfiguracijom i spaja rezultate.
    """

    audio = AudioSegment.from_file(str(audio_path))
    chunk_ms = int(chunk_seconds * 1000)

    tekstovi = []
    broj_chunkova = 0

    temp_dir = REZULTATI_DIR / "_temp_chunks"
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        for start_ms in range(0, len(audio), chunk_ms):
            end_ms = min(start_ms + chunk_ms, len(audio))
            chunk = audio[start_ms:end_ms]

            broj_chunkova += 1

            temp_path = (
                temp_dir /
                f"{Path(audio_path).stem}_chunk_{broj_chunkova:03d}.wav"
            )

            chunk.export(str(temp_path), format="wav")

            segments, info = model.transcribe(
                str(temp_path),
                language=language,
                task="transcribe",
                beam_size=beam_size,
                temperature=0.0,
                vad_filter=vad_filter,
                condition_on_previous_text=False,
                without_timestamps=False,
                initial_prompt=None
            )

            tekst_chunka = " ".join(
                segment.text.strip()
                for segment in segments
                if segment.text.strip()
            )

            if tekst_chunka:
                tekstovi.append(tekst_chunka)

            if temp_path.exists():
                temp_path.unlink()

    finally:
        try:
            temp_dir.rmdir()
        except OSError:
            pass

    finalni_tekst = " ".join(tekstovi).strip()

    return finalni_tekst, broj_chunkova


## Završno pokretanje na svih 40 snimaka

Merenje vremena obuhvata kompletan postupak obrade jednog snimka kroz sve njegove segmente. Model se učitava samo jednom pre početka benchmarka, tako da vreme učitavanja modela ne ulazi u vreme transkripcije pojedinačnog snimka.


In [ ]:
# ============================================================
# GLAVNI EKSPERIMENT — SVIH 40 SNIMAKA
# ============================================================

rezultati = []

print("=" * 100)
print("ZAVRŠNA VALIDACIJA INTEGRISANOG PROFILA")
print("Small INT8 + beam=5 + VAD ON + chunk=10 s")
print("=" * 100)

for i, item in enumerate(test_set, start=1):

    audio_path = Path(item["audio"])
    ref = item["ref"]

    trajanje = trajanje_audio(audio_path)

    print(f"\n[{i:02d}/40] {audio_path.name} ({trajanje:.2f} s)")

    start = time.perf_counter()

    hipoteza, broj_chunkova = transkribuj_u_chunkovima(
        model=model,
        audio_path=audio_path,
        chunk_seconds=CHUNK_SECONDS,
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=VAD_FILTER
    )

    vreme = time.perf_counter() - start

    ref_norm = normalizuj_tekst(ref)
    hyp_norm = normalizuj_tekst(hipoteza)

    if len(ref_norm) == 0:
        wer_vrednost = np.nan
    else:
        wer_vrednost = wer(ref_norm, hyp_norm) * 100

    rtf = vreme / trajanje if trajanje > 0 else np.nan

    rezultat = {
        "Broj": item.get("broj", i),
        "Fajl": audio_path.name,
        "Govornik": govornik_iz_imena(audio_path),

        "Model": MODEL_SIZE,
        "Kvantizacija": COMPUTE_TYPE,
        "Beam_size": BEAM_SIZE,
        "VAD": VAD_FILTER,
        "Chunk_s": CHUNK_SECONDS,

        "Trajanje_s": trajanje,
        "Broj_chunkova": broj_chunkova,

        "Referenca": ref,
        "Transkript": hipoteza,

        "WER_pct": wer_vrednost,
        "Vreme_s": vreme,
        "RTF": rtf
    }

    rezultati.append(rezultat)

    print(f"  chunkova: {broj_chunkova}")
    print(f"  WER:      {wer_vrednost:.2f}%")
    print(f"  vreme:    {vreme:.2f} s")
    print(f"  RTF:      {rtf:.4f}")

    print(f"  REF: {ref_norm[:150]}")
    print(f"  HYP: {hyp_norm[:150]}")


In [ ]:
# ============================================================
# TABELA REZULTATA I CSV
# ============================================================

df = pd.DataFrame(rezultati)

CSV_PATH = REZULTATI_DIR / "09_validacija_konacnog_profila.csv"

df.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Rezultati sačuvani:")
print(CSV_PATH)

display(
    df[
        [
            "Broj",
            "Fajl",
            "Govornik",
            "Trajanje_s",
            "Broj_chunkova",
            "WER_pct",
            "Vreme_s",
            "RTF"
        ]
    ].round(4)
)


In [ ]:
# ============================================================
# CORPUS WER
# ============================================================

sve_reference = " ".join(
    normalizuj_tekst(x)
    for x in df["Referenca"].tolist()
)

sve_hipoteze = " ".join(
    normalizuj_tekst(x)
    for x in df["Transkript"].tolist()
)

corpus_wer = wer(
    sve_reference,
    sve_hipoteze
) * 100

print(f"Corpus WER: {corpus_wer:.2f}%")


In [ ]:
# ============================================================
# KONAČNI REZIME ZA MASTER RAD
# ============================================================

rezime = pd.DataFrame([{
    "Konfiguracija": "Small INT8 + beam 5 + VAD + chunk 10 s",

    "Broj_snimaka": len(df),

    "Prosecan_WER_pct": df["WER_pct"].mean(),
    "Medijana_WER_pct": df["WER_pct"].median(),
    "Corpus_WER_pct": corpus_wer,

    "Prosecno_vreme_s": df["Vreme_s"].mean(),
    "Medijana_vremena_s": df["Vreme_s"].median(),
    "Ukupno_vreme_s": df["Vreme_s"].sum(),

    "Prosecan_RTF": df["RTF"].mean(),
    "Medijana_RTF": df["RTF"].median(),
    "Maksimalni_RTF": df["RTF"].max(),

    "Ukupno_chunkova": df["Broj_chunkova"].sum()
}])

display(rezime.round(4))

REZIME_PATH = (
    REZULTATI_DIR /
    "09_validacija_konacnog_profila_rezime.csv"
)

rezime.to_csv(
    REZIME_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSačuvan rezime:")
print(REZIME_PATH)


## Pregled najtežih slučajeva

Pored agregatnih metrika prikazuje se deset snimaka sa najvećim WER-om, zajedno sa referencom i dobijenim transkriptom. Ovaj pregled može se koristiti i za kvalitativnu proveru ekstremnih grešaka i mogućih halucinacija.


In [ ]:
# ============================================================
# 10 SNIMAKA SA NAJVEĆIM WER-om
# ============================================================

print("10 SNIMAKA SA NAJVEĆIM WER-om:")

najgori = (
    df.sort_values("WER_pct", ascending=False)
      .head(10)
)

display(
    najgori[
        [
            "Broj",
            "Fajl",
            "Govornik",
            "Trajanje_s",
            "Broj_chunkova",
            "WER_pct",
            "RTF"
        ]
    ].round(4)
)

for _, row in najgori.iterrows():

    print("\n" + "=" * 100)

    print(
        f"{row['Fajl']} | "
        f"WER={row['WER_pct']:.2f}% | "
        f"RTF={row['RTF']:.4f}"
    )

    print("\nREF:")
    print(row["Referenca"])

    print("\nHYP:")
    print(row["Transkript"])


## Tumačenje

Numerički zaključak za master rad unosi se tek nakon izvršavanja svih ćelija. Posebno treba navesti prosečan WER, corpus WER, prosečno vreme i prosečan RTF konačnog integrisanog profila.

Ovaj eksperiment predstavlja završnu validaciju profila kao celine i dopunjuje prethodne odvojene eksperimente za izbor modela, kvantizacije, Beam Search parametra, VAD-a i veličine segmenta.
